In [ ]:
# ============================================
# SUBSCRIPTION FRAUD DATASET GENERATOR
# 1,000 rows of realistic product/fraud data
# Designed for portfolio case studies
# ============================================

import pandas as pd
import numpy as np
import random
import uuid
from datetime import datetime, timedelta

# -----------------------------
# Reproducibility
# -----------------------------
random.seed(42)
np.random.seed(42)

# -----------------------------
# Config
# -----------------------------
N_ROWS = 1000
FRAUD_RATE = 0.15   # ~15% fraud, realistic enough for a portfolio dataset

TODAY = datetime(2026, 3, 1)

# -----------------------------
# ASN Providers
# -----------------------------
RESIDENTIAL_ASNS = [
    ("AS7922", "Comcast", "residential"),
    ("AS7018", "AT&T", "residential"),
    ("AS21928", "T-Mobile", "residential"),
    ("AS20057", "Verizon", "residential"),
    ("AS10796", "Charter", "residential"),
]

CLOUD_ASNS = [
    ("AS16509", "Amazon AWS", "cloud"),
    ("AS15169", "Google Cloud", "cloud"),
    ("AS8075", "Microsoft Azure", "cloud"),
    ("AS14618", "Amazon-AES", "cloud"),
    ("AS13335", "Cloudflare", "cloud"),
]

VPN_ASNS = [
    ("AS9009", "M247 VPN", "vpn"),
    ("AS20473", "Vultr VPN", "vpn"),
    ("AS63023", "GTHost VPN", "vpn"),
    ("AS212238", "Datacamp VPN", "vpn"),
]

ALL_ASNS = RESIDENTIAL_ASNS + CLOUD_ASNS + VPN_ASNS

# -----------------------------
# Countries
# -----------------------------
LOW_RISK_COUNTRIES = ["US", "CA", "UK"]
HIGHER_RISK_COUNTRIES = ["NG", "BR", "PH", "ID", "VN"]

# -----------------------------
# Subscription Plans
# -----------------------------
PLANS = ["trial", "monthly", "annual"]

# -----------------------------
# Payment status options
# -----------------------------
LEGIT_PAYMENT_STATUSES = ["paid", "paid", "paid", "paid", "failed", "refunded"]
FRAUD_PAYMENT_STATUSES = ["paid", "failed", "refunded", "chargeback", "refunded", "chargeback"]

# -----------------------------
# SSL fingerprints
# These are synthetic but plausible-looking
# -----------------------------
COMMON_SSL_FINGERPRINTS = [
    "TLS_771_4865-4866-4867_chrome_win",
    "TLS_771_4865-4866_safari_ios",
    "TLS_771_4865-49195_chrome_mac",
    "TLS_772_4865-4866-4867_edge_win",
    "TLS_772_4865-49196_firefox_linux",
]

BOT_SSL_FINGERPRINTS = [
    "TLS_771_4865_headless_chrome",
    "TLS_771_49195_python_requests",
    "TLS_772_4865_playwright_bot",
    "TLS_771_4866_selenium_driver",
    "TLS_772_4865_puppeteer_bot",
]

# -----------------------------
# Utility functions
# -----------------------------
def random_date_within_last_days(max_days=365):
    days_ago = random.randint(0, max_days)
    return TODAY - timedelta(days=days_ago)

def generate_transaction_id(i):
    return f"TXN_{i:05d}"

def generate_user_id(i):
    return f"USR_{i:05d}"

def generate_device_id():
    return f"DVC_{uuid.uuid4().hex[:10].upper()}"

def generate_datr_token():
    return f"DATR_{uuid.uuid4().hex[:16]}"

def generate_ip_address():
    return ".".join(str(random.randint(1, 254)) for _ in range(4))

def choose_asn(network_type=None):
    if network_type == "residential":
        return random.choice(RESIDENTIAL_ASNS)
    elif network_type == "cloud":
        return random.choice(CLOUD_ASNS)
    elif network_type == "vpn":
        return random.choice(VPN_ASNS)
    else:
        return random.choice(ALL_ASNS)

def choose_country(network_type="residential", fraud=False):
    if fraud:
        if network_type in ["cloud", "vpn"]:
            return random.choices(
                HIGHER_RISK_COUNTRIES + LOW_RISK_COUNTRIES,
                weights=[20, 18, 14, 14, 10, 10, 8, 6],
                k=1
            )[0]
        return random.choices(
            LOW_RISK_COUNTRIES + HIGHER_RISK_COUNTRIES,
            weights=[40, 15, 10, 12, 8, 5, 5, 5],
            k=1
        )[0]
    else:
        return random.choices(
            LOW_RISK_COUNTRIES + HIGHER_RISK_COUNTRIES,
            weights=[65, 15, 10, 3, 2, 2, 1, 2],
            k=1
        )[0]

def weighted_plan_choice(fraud=False):
    if fraud:
        return random.choices(PLANS, weights=[45, 45, 10], k=1)[0]
    return random.choices(PLANS, weights=[25, 55, 20], k=1)[0]

# -----------------------------
# Pre-build device farms for fraud
# Some devices reused across many accounts
# -----------------------------
fraud_device_farms = [generate_device_id() for _ in range(18)]

# each device farm will be reused several times
fraud_device_pool = []
for dvc in fraud_device_farms:
    reuse_count = random.randint(4, 10)
    fraud_device_pool.extend([dvc] * reuse_count)

random.shuffle(fraud_device_pool)

# -----------------------------
# Fraud type templates
# -----------------------------
FRAUD_TYPES = [
    "device_farm",
    "trial_abuse",
    "refund_abuse",
    "stolen_card",
]

# -----------------------------
# Generate raw rows
# -----------------------------
rows = []

n_fraud = int(N_ROWS * FRAUD_RATE)
n_legit = N_ROWS - n_fraud

# ----- Generate legit rows -----
for i in range(1, n_legit + 1):
    transaction_id = generate_transaction_id(i)
    user_id = generate_user_id(i)
    device_id = generate_device_id()
    datr_token = generate_datr_token()
    signup_date = random_date_within_last_days(365)
    account_age_days = (TODAY - signup_date).days

    asn, asn_provider, asn_type = choose_asn("residential")
    ip_country = choose_country(network_type="residential", fraud=False)
    ip_address = generate_ip_address()

    ssl_fingerprint = random.choice(COMMON_SSL_FINGERPRINTS)
    subscription_plan = weighted_plan_choice(fraud=False)
    trial_used_before = np.random.choice([0, 1], p=[0.96, 0.04])

    # Legit refund behavior
    refund_count_90d = np.random.choice([0, 1, 2], p=[0.88, 0.10, 0.02])

    payment_status = random.choice(LEGIT_PAYMENT_STATUSES)

    # annual users less likely to refund/failed
    if subscription_plan == "annual" and payment_status in ["failed", "refunded"]:
        payment_status = "paid"

    # older legit users usually not flagged
    fraud_label = 0
    fraud_type = "legit"

    rows.append({
        "transaction_id": transaction_id,
        "user_id": user_id,
        "signup_date": signup_date.date(),
        "device_id": device_id,
        "datr_token": datr_token,
        "ssl_fingerprint": ssl_fingerprint,
        "ip_address": ip_address,
        "asn": asn,
        "asn_provider": asn_provider,
        "asn_type": asn_type,
        "ip_country": ip_country,
        "subscription_plan": subscription_plan,
        "payment_status": payment_status,
        "refund_count_90d": int(refund_count_90d),
        "trial_used_before": int(trial_used_before),
        "account_age_days": int(account_age_days),
        "fraud_type": fraud_type,
        "fraud_label": fraud_label
    })

# ----- Generate fraud rows -----
for i in range(n_legit + 1, N_ROWS + 1):
    transaction_id = generate_transaction_id(i)
    user_id = generate_user_id(i)

    fraud_type = random.choices(
        FRAUD_TYPES,
        weights=[35, 25, 20, 20],
        k=1
    )[0]

    # Device behavior by fraud type
    if fraud_type == "device_farm":
        device_id = random.choice(fraud_device_pool)
        datr_token = f"DATR_SHARED_{device_id[-6:]}"
        ssl_fingerprint = random.choice(BOT_SSL_FINGERPRINTS)
        network_choice = random.choices(["cloud", "vpn"], weights=[60, 40], k=1)[0]
        signup_date = random_date_within_last_days(30)
        subscription_plan = random.choice(["trial", "monthly"])
        trial_used_before = np.random.choice([0, 1], p=[0.25, 0.75])
        refund_count_90d = np.random.choice([0, 1, 2], p=[0.60, 0.25, 0.15])
        payment_status = random.choice(["paid", "failed", "refunded"])

    elif fraud_type == "trial_abuse":
        device_id = random.choice(fraud_device_pool)
        datr_token = f"DATR_TRIAL_{uuid.uuid4().hex[:10]}"
        ssl_fingerprint = random.choice(COMMON_SSL_FINGERPRINTS + BOT_SSL_FINGERPRINTS)
        network_choice = random.choices(["residential", "vpn", "cloud"], weights=[20, 40, 40], k=1)[0]
        signup_date = random_date_within_last_days(14)
        subscription_plan = "trial"
        trial_used_before = 1
        refund_count_90d = np.random.choice([0, 1, 2], p=[0.70, 0.20, 0.10])
        payment_status = random.choice(["paid", "failed", "refunded"])

    elif fraud_type == "refund_abuse":
        device_id = generate_device_id() if random.random() < 0.35 else random.choice(fraud_device_pool)
        datr_token = f"DATR_REFUND_{uuid.uuid4().hex[:10]}"
        ssl_fingerprint = random.choice(COMMON_SSL_FINGERPRINTS)
        network_choice = random.choices(["residential", "vpn"], weights=[40, 60], k=1)[0]
        signup_date = random_date_within_last_days(90)
        subscription_plan = random.choice(["monthly", "annual"])
        trial_used_before = np.random.choice([0, 1], p=[0.55, 0.45])
        refund_count_90d = np.random.choice([2, 3, 4, 5], p=[0.30, 0.35, 0.25, 0.10])
        payment_status = random.choice(["refunded", "refunded", "chargeback", "paid"])

    else:  # stolen_card
        device_id = generate_device_id() if random.random() < 0.50 else random.choice(fraud_device_pool)
        datr_token = f"DATR_CARD_{uuid.uuid4().hex[:10]}"
        ssl_fingerprint = random.choice(COMMON_SSL_FINGERPRINTS + BOT_SSL_FINGERPRINTS)
        network_choice = random.choices(["cloud", "vpn", "residential"], weights=[45, 35, 20], k=1)[0]
        signup_date = random_date_within_last_days(10)
        subscription_plan = random.choice(["monthly", "annual"])
        trial_used_before = np.random.choice([0, 1], p=[0.85, 0.15])
        refund_count_90d = np.random.choice([0, 1, 2], p=[0.65, 0.20, 0.15])
        payment_status = random.choice(["failed", "chargeback", "paid", "chargeback"])

    account_age_days = (TODAY - signup_date).days
    asn, asn_provider, asn_type = choose_asn(network_choice)
    ip_country = choose_country(network_type=asn_type, fraud=True)
    ip_address = generate_ip_address()
    fraud_label = 1

    rows.append({
        "transaction_id": transaction_id,
        "user_id": user_id,
        "signup_date": signup_date.date(),
        "device_id": device_id,
        "datr_token": datr_token,
        "ssl_fingerprint": ssl_fingerprint,
        "ip_address": ip_address,
        "asn": asn,
        "asn_provider": asn_provider,
        "asn_type": asn_type,
        "ip_country": ip_country,
        "subscription_plan": subscription_plan,
        "payment_status": payment_status,
        "refund_count_90d": int(refund_count_90d),
        "trial_used_before": int(trial_used_before),
        "account_age_days": int(account_age_days),
        "fraud_type": fraud_type,
        "fraud_label": fraud_label
    })

# -----------------------------
# Build DataFrame
# -----------------------------
df = pd.DataFrame(rows)

# -----------------------------
# Derived features
# These make the data look more analyst-ready
# -----------------------------
device_counts = df.groupby("device_id")["user_id"].transform("count")
df["accounts_per_device"] = device_counts

# velocity-like simple feature:
# more suspicious when many accounts share a device
df["high_device_reuse_flag"] = (df["accounts_per_device"] >= 4).astype(int)

# suspicious country flag
df["high_risk_country_flag"] = df["ip_country"].isin(HIGHER_RISK_COUNTRIES).astype(int)

# suspicious network flag
df["non_residential_asn_flag"] = (df["asn_type"] != "residential").astype(int)

# chargeback/refund flags
df["chargeback_flag"] = (df["payment_status"] == "chargeback").astype(int)
df["refunded_flag"] = (df["payment_status"] == "refunded").astype(int)

# Simple rule-based risk score
df["risk_score"] = (
    df["high_device_reuse_flag"] * 30
    + (df["refund_count_90d"] >= 2).astype(int) * 20
    + df["trial_used_before"] * 15
    + df["high_risk_country_flag"] * 10
    + df["non_residential_asn_flag"] * 15
    + df["chargeback_flag"] * 30
    + (df["account_age_days"] <= 14).astype(int) * 10
)

# Optional rule-based review bucket
def assign_review_bucket(score):
    if score >= 70:
        return "high_risk"
    elif score >= 40:
        return "medium_risk"
    else:
        return "low_risk"

df["review_bucket"] = df["risk_score"].apply(assign_review_bucket)

# -----------------------------
# Reorder columns cleanly
# -----------------------------
column_order = [
    "transaction_id",
    "user_id",
    "signup_date",
    "account_age_days",
    "device_id",
    "datr_token",
    "ssl_fingerprint",
    "accounts_per_device",
    "high_device_reuse_flag",
    "ip_address",
    "asn",
    "asn_provider",
    "asn_type",
    "non_residential_asn_flag",
    "ip_country",
    "high_risk_country_flag",
    "subscription_plan",
    "trial_used_before",
    "payment_status",
    "refund_count_90d",
    "refunded_flag",
    "chargeback_flag",
    "risk_score",
    "review_bucket",
    "fraud_type",
    "fraud_label"
]

df = df[column_order]

# -----------------------------
# Shuffle rows so fraud isn't grouped at bottom
# -----------------------------
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# -----------------------------
# Save CSV
# -----------------------------
output_file = "subscription_fraud_dataset.csv"
df.to_csv(output_file, index=False)

# -----------------------------
# Quick checks
# -----------------------------
print(f"Dataset saved as: {output_file}")
print("\nShape:")
print(df.shape)

print("\nFraud label distribution:")
print(df["fraud_label"].value_counts())

print("\nFraud type distribution:")
print(df["fraud_type"].value_counts())

print("\nPayment status by fraud label:")
print(pd.crosstab(df["payment_status"], df["fraud_label"], normalize="columns").round(3))

print("\nTop reused devices:")
print(df["device_id"].value_counts().head(10))

print("\nPreview:")
display(df.head(10))

Dataset saved as: subscription_fraud_dataset.csv

Shape:
(1000, 26)

Fraud label distribution:
fraud_label
0    850
1    150
Name: count, dtype: int64

Fraud type distribution:
fraud_type
legit           850
trial_abuse      49
device_farm      47
refund_abuse     35
stolen_card      19
Name: count, dtype: int64

Payment status by fraud label:
fraud_label         0      1
payment_status              
chargeback      0.000  0.107
failed          0.131  0.227
paid            0.746  0.373
refunded        0.124  0.293

Top reused devices:
device_id
DVC_CE27447E7E    12
DVC_74F078F152    11
DVC_FAD9D8DB47    11
DVC_E30306FB00     9
DVC_505E5C1C4E     9
DVC_329F059288     8
DVC_9B413A4795     8
DVC_77EF3722A7     8
DVC_E00E56B119     7
DVC_8B0521A103     7
Name: count, dtype: int64

Preview:


,transaction_id,user_id,signup_date,account_age_days,device_id,datr_token,ssl_fingerprint,accounts_per_device,high_device_reuse_flag,ip_address,...,subscription_plan,trial_used_before,payment_status,refund_count_90d,refunded_flag,chargeback_flag,risk_score,review_bucket,fraud_type,fraud_label
0,TXN_00522,USR_00522,2025-04-02,333,DVC_032EBF5B32,DATR_8aea82b2edc84ae4,TLS_772_4865-49196_firefox_linux,1,0,22.243.25.43,...,monthly,0,refunded,0,1,0,0,low_risk,legit,0
1,TXN_00738,USR_00738,2025-03-28,338,DVC_CE07500BE3,DATR_7f29fcc291b347aa,TLS_771_4865-4866-4867_chrome_win,1,0,210.36.228.149,...,monthly,0,refunded,0,1,0,0,low_risk,legit,0
2,TXN_00741,USR_00741,2026-01-15,45,DVC_11B28050FB,DATR_ebbe7b95bfab4638,TLS_771_4865-4866_safari_ios,1,0,253.131.104.217,...,annual,0,paid,0,0,0,0,low_risk,legit,0
3,TXN_00661,USR_00661,2025-09-03,179,DVC_9FBD5F9D8C,DATR_ddf5749cd72d408c,TLS_772_4865-4866-4867_edge_win,1,0,206.114.160.38,...,trial,0,paid,0,0,0,0,low_risk,legit,0
4,TXN_00412,USR_00412,2025-03-14,352,DVC_C4F358584B,DATR_5dd73ad2df434324,TLS_772_4865-49196_firefox_linux,1,0,187.1.243.180,...,monthly,0,failed,0,0,0,0,low_risk,legit,0
5,TXN_00679,USR_00679,2025-05-15,290,DVC_6508C6D457,DATR_9376ea25187b4b0e,TLS_771_4865-49195_chrome_mac,1,0,60.60.176.7,...,trial,0,paid,0,0,0,0,low_risk,legit,0
6,TXN_00627,USR_00627,2025-07-01,243,DVC_99690CBD49,DATR_796c027b77d2480d,TLS_772_4865-4866-4867_edge_win,1,0,198.23.64.41,...,trial,0,failed,0,0,0,0,low_risk,legit,0
7,TXN_00514,USR_00514,2025-05-21,284,DVC_C69C61893C,DATR_c1e16be1eb03422d,TLS_771_4865-49195_chrome_mac,1,0,150.217.66.2,...,monthly,0,paid,0,0,0,0,low_risk,legit,0
8,TXN_00860,USR_00860,2025-12-16,75,DVC_74F078F152,DATR_REFUND_62fe826212,TLS_772_4865-4866-4867_edge_win,11,1,91.252.98.109,...,monthly,1,paid,5,0,0,65,medium_risk,refund_abuse,1
9,TXN_00137,USR_00137,2025-10-17,135,DVC_411EEA7453,DATR_a300386ca06b4d0a,TLS_771_4865-4866-4867_chrome_win,1,0,108.135.30.191,...,monthly,0,paid,1,0,0,0,low_risk,legit,0


In [ ]:
from google.colab import files
files.download("subscription_fraud_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>